In [ ]:
def require_role(*allowed_roles: UserGroupEnum):  # ← outer function, NOT a decorator
                                                  #   because it takes roles, not a callable
    def role_checker(
        current_user: Annotated[User, Depends(get_current_user)],  # ← FastAPI injects this
    ) -> User:
        # 'allowed_roles' is a FREE VARIABLE — lives in the enclosing scope
        # role_checker CLOSES OVER it → this is the closure
        if not any(current_user.has_group(role) for role in allowed_roles):
            allowed = ", ".join(role.value for role in allowed_roles)
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Access denied. Required role(s): {allowed}",
            )
        return current_user

    return role_checker  # ← returns a callable, not a decorated function

**Decorator vs Closure — The Precise Distinction**
```shell
DECORATOR pattern:                    YOUR pattern (Closure / Dependency Factory):

takes a CALLABLE as argument          takes CONFIGURATION as argument
  ↓                                     ↓
def decorator(func):                  def require_role(*allowed_roles):

wraps it with another callable        returns a plain callable (no wrapping)
  ↓                                     ↓
  def wrapper(*args, **kwargs):         def role_checker(current_user):
      return func(*args, **kwargs)          return current_user
  return wrapper                        return role_checker

used with @syntax or func=dec(func)   used as Depends(require_role(ADMIN))
```
A decorator **IS** a closure too — but not all closures are decorators. The defining trait of a decorator is **taking a callable and returning a callable that wraps it**. Yours takes roles and returns a dependency. Nailing this distinction in an interview is a strong signal.

---

**How Your Closure Memory Works**
```py
# Call 1
admin_checker = require_role(UserGroupEnum.ADMIN)
# A closure object is created in memory:
# role_checker + cell{ allowed_roles = (ADMIN,) }

# Call 2
editor_checker = require_role(UserGroupEnum.ADMIN, UserGroupEnum.EDITOR)
# A NEW closure object:
# role_checker + cell{ allowed_roles = (ADMIN, EDITOR) }

# Each Depends(...) gets its OWN closure with its OWN frozen allowed_roles
# This is exactly what makes it safe to reuse across many routes

# You can inspect the closure cells yourself:
checker = require_role(UserGroupEnum.ADMIN)
print(checker.__closure__)          # (<cell at 0x...>,)
print(checker.__closure__[0].cell_contents)  # (UserGroupEnum.ADMIN,)
print(checker.__code__.co_freevars)          # ('allowed_roles',)
```